# Librerias

In [1]:
import os, re, json, unicodedata, sys
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_constantes(inicio=None):
    """Devuelve la carpeta que contiene constantes.py.

    Busca desde la ubicación del cuaderno hacia arriba. En cada nivel prueba
    la carpeta misma y su subcarpeta 'Bloque A', que es donde vive hoy el
    módulo. Así el cuaderno corre en cualquier equipo sin rutas escritas a mano.
    """
    inicio = Path(inicio or Path.cwd()).resolve()
    for base in [inicio, *inicio.parents]:
        for candidata in (base, base / "Bloque A"):
            if (candidata / "constantes.py").is_file():
                return candidata
    raise FileNotFoundError(
        "No se encontró constantes.py subiendo desde " + str(inicio)
    )


RUTA_CONSTANTES = localizar_constantes()
if str(RUTA_CONSTANTES) not in sys.path:
    sys.path.insert(0, str(RUTA_CONSTANTES))

import constantes

# Se imprime cuál se cargó: si hubiera una copia vieja del módulo en otra
# carpeta, el error aparece aquí y no diez celdas más abajo con números raros.
print("constantes.py cargado desde:", RUTA_CONSTANTES / "constantes.py")
print("RUTA_PROYECTO:            ", constantes.RUTA_PROYECTO)
print("Salidas de este cuaderno: ", constantes.ruta_entregable("B4"))

constantes.py cargado desde: F:\Tesis\Bloque A\constantes.py
RUTA_PROYECTO:             F:\Tesis
Salidas de este cuaderno:  F:\Tesis\Bloque B\Entregable B4


# B4 Reconciliación y completitud de atributos de ruteo

In [2]:
# ORDEN DEL BLOQUE B (lineal, sin ciclos):
#   B1    empareja por distancia; clasifica; deja 6 casos sin verificar
#   B2    mide la trama urbana y resuelve casos con manzana catastral (UAECD)
#   B2.1  aplica los veredictos y contrasta contra B1
#   B3    empareja por puntaje combinado
#   B3.1  evalúa ese emparejamiento contra la manzana catastral
#   B4    reconcilia atributos y caracteriza su completitud
#
# ALCANCE
# El entregable pedido es una tabla de concordancia por atributo entre las dos
# fuentes. Se produce (sección 1), y se acompaña de la medición que la hace
# interpretable: qué proporción de estaciones tiene atributos comparables
# (sección 2), en qué dirección fluye la información (sección 3) y qué
# fracción del inventario consolidado queda utilizable para el ruteo
# (sección 4). La concordancia sobre un puñado de pares no sostiene ninguna
# afirmación por sí sola; la completitud sí, y además es la que condiciona el
# objetivo específico 5.
#
# Este cuaderno no modifica salidas de bloques anteriores.
# =============================================================================

RUTA_A2   = Path(constantes.ruta_entregable("A2"))
RUTA_A3   = Path(constantes.ruta_entregable("A3"))
RUTA_B2_1 = Path(constantes.ruta_entregable("B2.1"))
RUTA_B3   = Path(constantes.ruta_entregable("B3"))
RUTA_OUT  = Path(constantes.ruta_entregable("B4"))
RUTA_OUT.mkdir(parents=True, exist_ok=True)

# Atributos que determinan la factibilidad de una ruta, con su nombre en cada
# fuente. El orden es el de importancia declarada en el anteproyecto.
#
# En OSM, «potencia» y «operador» NO son una sola columna. valor_osm() los
# resuelve así:
#   potencia -> la clave general o, si falta, el máximo de socket:*:output.
#               OSM puede declarar la potencia por conector; leer solo la
#               clave general reportaba 0 de 53 cuando hay 1 (Tesla, Bogotá).
#   operador -> operador, marca o red, en ese orden. Es la misma regla de
#               constantes.cargar_fuentes_estaciones(), que usó B3. Leer solo
#               «operador» dejaba sin operador a los nodos marcados EVX.
ATRIBUTOS = [
    ("potencia",  "potencia",     "potencia_kw_max",
     "kW disponibles; determina el tiempo de carga"),
    ("conector",  "sockets_json", "conectores_lista",
     "compatibilidad física con el vehículo"),
    ("acceso",    "acceso",       "tipo_uso",
     "si la estación es de uso público"),
    ("cobro",     "cobro",        "costo_uso",
     "condiciones económicas de uso"),
    ("capacidad", "capacidad",    "n_puntos_declarados",
     "número de puntos de carga simultáneos"),
    ("operador",  "operador",     "operador",
     "responsable de la operación"),
]

print("B4 · Reconciliación y completitud de atributos de ruteo")
print(f"  Salidas en: {RUTA_OUT}")

B4 · Reconciliación y completitud de atributos de ruteo
  Salidas en: F:\Tesis\Bloque B\Entregable B4


In [3]:
# =============================================================================
# Procedencia de los insumos
#
# B4 consume dos clases de archivo y cada una se verifica contra un registro
# distinto de B3:
#
#   1. INSUMOS que B3 también consumió (A2, A3, B2.1). Se comparan contra
#      B3_manifiesto_insumos.json: si no coinciden, B4 estaría leyendo otra
#      versión de los datos que B3 emparejó.
#
#   2. SALIDAS de B3 (pares, alias, parámetros). Se comparan contra
#      B3_huellas_salidas.json: si no coinciden, B3 se volvió a correr (o se
#      editó un archivo a mano) después de registrar sus huellas.
#
# El segundo control es el que faltaba: la versión anterior de B4 corrió con
# 12 pares y 17 alias cuando B3 ya tenía 13 y 21, y nada lo detectó.
# =============================================================================

ARCH_MANIFIESTO_B3 = constantes.exigir_archivo(
    constantes.ruta_entregable("B3", "B3_manifiesto_insumos.json"))
ARCH_HUELLAS_B3 = constantes.exigir_archivo(
    constantes.ruta_entregable("B3", "B3_huellas_salidas.json"))

with open(ARCH_MANIFIESTO_B3, encoding="utf-8") as fh:
    MANIFIESTO_B3 = json.load(fh)
with open(ARCH_HUELLAS_B3, encoding="utf-8") as fh:
    HUELLAS_B3 = json.load(fh)

# La clave es el NOMBRE DEL ARCHIVO, igual que en los manifiestos de B2 y B3,
# para que la comparación sea directa.
INSUMOS_COMPARTIDOS = {
    "A2_estaciones_OSM_Bogota.geojson":
        RUTA_A2 / "A2_estaciones_OSM_Bogota.geojson",
    "A2_estaciones_OSM_Medellin.geojson":
        RUTA_A2 / "A2_estaciones_OSM_Medellin.geojson",
    "A3_OpenChargeMap_Bogota_estaciones.csv":
        RUTA_A3 / "A3_OpenChargeMap_Bogota_estaciones.csv",
    "A3_OpenChargeMap_Medellin_estaciones.csv":
        RUTA_A3 / "A3_OpenChargeMap_Medellin_estaciones.csv",
    "B2_1_clasificacion_verificada.csv":
        RUTA_B2_1 / "B2_1_clasificacion_verificada.csv",
}

SALIDAS_B3_CONSUMIDAS = {
    "B3_pares_puntaje_combinado.csv":
        RUTA_B3 / "B3_pares_puntaje_combinado.csv",
    "B3_diccionario_alias_operador.csv":
        RUTA_B3 / "B3_diccionario_alias_operador.csv",
    "B3_parametros.json":
        RUTA_B3 / "B3_parametros.json",
}


def verificar(archivos, registro, nombre_registro):
    """Calcula el SHA-256 de cada archivo y lo compara con el registro.

    A diferencia de B3, aquí un archivo AUSENTE del registro también es falla:
    todos los que B4 consume fueron producidos o consumidos por B3, así que
    no hay ninguno «posterior» que se pueda registrar sin comparar.
    """
    hashes, fallas = {}, []
    for nombre, ruta in archivos.items():
        constantes.exigir_archivo(ruta)
        calculado = constantes.sha256_archivo(ruta)
        hashes[nombre] = calculado
        registrado = registro.get(nombre)
        if registrado is None:
            estado = f"NO FIGURA en {nombre_registro}"
            fallas.append(nombre)
        elif registrado == calculado:
            estado = f"coincide con {nombre_registro}"
        else:
            estado = f"NO COINCIDE con {nombre_registro}"
            fallas.append(nombre)
        print(f"  {nombre}")
        print(f"      {calculado[:16]}…  {estado}")
    return hashes, fallas


print("PROCEDENCIA DE LOS INSUMOS")
print("=" * 78)
print("Insumos compartidos con B3:")
HASHES_COMPARTIDOS, fallas_insumos = verificar(
    INSUMOS_COMPARTIDOS, MANIFIESTO_B3, "manifiesto de B3")

print("\nSalidas de B3 consumidas por B4:")
HASHES_SALIDAS_B3, fallas_salidas = verificar(
    SALIDAS_B3_CONSUMIDAS, HUELLAS_B3, "huellas de B3")

if fallas_insumos:
    raise RuntimeError(
        "Los insumos cambiaron respecto de los que consumió B3: "
        + ", ".join(fallas_insumos)
        + ".\nLos pares de B3 describen otra versión de los datos. Si la "
        "regeneración fue intencional, hay que correr la cadena completa "
        "hacia adelante: B1, B2, B2.1, B3 y recién después B4."
    )

if fallas_salidas:
    raise RuntimeError(
        "Las salidas de B3 no coinciden con las huellas que B3 registró: "
        + ", ".join(fallas_salidas)
        + ".\nB3 se corrió de nuevo sin terminar, o un archivo se editó a "
        "mano. Hay que correr B3 completo antes de B4."
    )

# --- Crudos congelados, heredados de B3 --------------------------------------
# Se arrastran sin recalcular: B2 los validó contra los archivos congelados de
# A2 y A3, y B3 los heredó de B2. Son el eslabón final de la cadena.
CRUDOS_HEREDADOS = {k: v for k, v in MANIFIESTO_B3.items()
                    if k.startswith("crudo::")}

if not CRUDOS_HEREDADOS:
    raise RuntimeError(
        "El manifiesto de B3 no trae entradas 'crudo::'. Sin ellas la cadena "
        "de procedencia queda cortada y no llega a los archivos congelados."
    )

# Lo que B4 escribirá en su propio manifiesto al final del cuaderno.
HASHES_INSUMOS = {**HASHES_COMPARTIDOS, **HASHES_SALIDAS_B3}

print(f"\n  Archivos verificados: {len(HASHES_INSUMOS)} "
      f"({len(HASHES_COMPARTIDOS)} insumos + {len(HASHES_SALIDAS_B3)} salidas de B3)")
print(f"  Crudos congelados heredados de B3: {len(CRUDOS_HEREDADOS)}")
for n in sorted(CRUDOS_HEREDADOS):
    print(f"      {n.removeprefix('crudo::')}")
print(f"\n  Fecha de corte del inventario: {constantes.FECHA_CORTE_LOCAL}")

PROCEDENCIA DE LOS INSUMOS
Insumos compartidos con B3:
  A2_estaciones_OSM_Bogota.geojson
      99f132a15e2a87e9…  coincide con manifiesto de B3
  A2_estaciones_OSM_Medellin.geojson
      69a8c0bafbf86fce…  coincide con manifiesto de B3
  A3_OpenChargeMap_Bogota_estaciones.csv
      cb12e624d5aa1856…  coincide con manifiesto de B3
  A3_OpenChargeMap_Medellin_estaciones.csv
      02767749ab5e1857…  coincide con manifiesto de B3
  B2_1_clasificacion_verificada.csv
      b0a2e926cf09b73c…  coincide con manifiesto de B3

Salidas de B3 consumidas por B4:
  B3_pares_puntaje_combinado.csv
      6c288bede04300c0…  coincide con huellas de B3
  B3_diccionario_alias_operador.csv
      ca14d795974ce070…  coincide con huellas de B3
  B3_parametros.json
      20fc689e8ab3c1e4…  coincide con huellas de B3

  Archivos verificados: 8 (5 insumos + 3 salidas de B3)
  Crudos congelados heredados de B3: 3
      A2_osm_crudo_Bogota_2026-08-29
      A2_osm_crudo_Medellin_2026-08-29
      A3_ocm_crudo_Colombi

# Utilidades

In [4]:
def declarado(v):
    """True si la fuente declara un valor real para el atributo.
    Un centinela no es un dato: OpenChargeMap escribe «Unknown» donde no
    conoce el conector, «(Unknown Operator)» donde no conoce al operador y
    «Application» en costo_uso cuando la tarifa depende de la aplicación;
    OpenStreetMap deja diccionarios vacíos «{}» en sockets_json.
    La lista vive en constantes.CENTINELAS."""
    return not constantes.es_centinela(v)


def normalizar(t):
    """Normalización común (constantes.normalizar_texto), con los centinelas
    convertidos en cadena vacía."""
    return constantes.normalizar_texto(t) if declarado(t) else ""


def numero(v):
    """Convierte a número una cantidad declarada («2», 2, «325 kW»).
    Devuelve None si no hay número. Sirve para comparar capacidades como
    cantidades y no como cadenas: «2» y «2.0» son el mismo valor."""
    if not declarado(v):
        return None
    m = re.search(r"\d+(?:[.,]\d+)?", str(v))
    return float(m.group().replace(",", ".")) if m else None


def a_diccionario(v):
    """sockets_json llega como dict desde el GeoJSON, pero puede llegar como
    texto si alguien lo relee desde CSV. Se acepta de las dos formas."""
    if isinstance(v, dict):
        return v
    if not declarado(v):
        return {}
    try:
        return json.loads(v)
    except (TypeError, ValueError):
        return {}


# =============================================================================
# Lectura por fuente
#
# ATRIBUTOS guarda un nombre de columna por fuente, pero en OSM dos atributos
# no viven en una sola columna. Toda sección del cuaderno lee a través de
# valor_osm() y valor_ocm(); ninguna hace o.get(clave) directo. Así la regla
# se define una vez y no puede divergir entre secciones, que es lo que pasaba
# antes con el operador.
# =============================================================================

def potencia_osm(p):
    """Potencia declarada en OSM, en kW.
    Primero la clave general; si falta, el máximo de las claves
    socket:<tipo>:output. El Tesla Supercharger de Bogotá solo la declara
    por conector (325 kW): leer únicamente la clave general reportaba que
    ninguna estación OSM declara potencia."""
    if declarado(p.get("potencia")):
        return numero(p.get("potencia"))
    kw = [numero(v) for k, v in a_diccionario(p.get("sockets_json")).items()
          if k.startswith("socket:") and k.endswith(":output")]
    kw = [x for x in kw if x is not None]
    return max(kw) if kw else None


def operador_osm(p):
    """Operador en OSM: operator, brand o network, en ese orden.
    Es la misma regla de constantes.cargar_fuentes_estaciones(), que usó B3.
    Leer solo «operador» dejaba sin operador a los nodos marcados EVX."""
    for clave in ("operador", "marca", "red"):
        if declarado(p.get(clave)):
            return p.get(clave)
    return None


def valor_osm(o, clave):
    if clave == "potencia":
        return potencia_osm(o)
    if clave == "operador":
        return operador_osm(o)
    return o.get(clave)


def valor_ocm(m, clave):
    return getattr(m, clave, None)


# =============================================================================
# Vocabularios comunes
#
# Las dos fuentes codifican el mismo hecho con notaciones distintas. Comparar
# las cadenas en bruto mide diferencias de codificación, no discrepancias de
# contenido: «fee=no» en OpenStreetMap y «Free» en OpenChargeMap significan lo
# mismo. Cada atributo se traduce a un vocabulario común antes de comparar, y
# los términos sin equivalencia se reportan en vez de contarse como ausentes.
# =============================================================================

MAPA_CONECTOR = {
    # OpenStreetMap — claves socket:*
    "type1": "TYPE1", "type1_combo": "CCS1", "type2": "TYPE2",
    "type2_combo": "CCS2", "type2_cable": "TYPE2", "chademo": "CHADEMO",
    "gb-t20234": "GBT", "gb_t20234": "GBT",
    # Type E (CEE 7/5, francés) y Schuko (CEE 7/4, Type F) son enchufes
    # distintos. La versión anterior los fusionaba.
    "typee": "TYPEE", "schuko": "SCHUKO",
    "nema_5_15": "NEMA515", "nema5_15": "NEMA515",
    "nema_6_15": "NEMA615", "nema_14_50": "NEMA1450",
    "tesla_supercharger": "TESLA",
    # Respaldado por la propia OCM: el Tesla de Unicentro Medellín (491538)
    # se declara «CCS (Type 2)».
    "tesla_supercharger_ccs": "CCS2",
    # OpenChargeMap — denominaciones comerciales, ya normalizadas
    "type 1 j1772": "TYPE1", "j1772": "TYPE1", "type 1": "TYPE1",
    "type 2 socket only": "TYPE2", "type 2 tethered connector": "TYPE2",
    "type 2": "TYPE2", "mennekes": "TYPE2",
    "ccs type 1": "CCS1", "ccs type 2": "CCS2",
    "chademo": "CHADEMO",
    "gb t dc gb t 20234 3": "GBT", "gb t ac gb t 20234 2": "GBT",
    "gb t dc": "GBT", "gb t": "GBT",
    "nema 5 15r": "NEMA515", "nema 5 20r": "NEMA515",
    "nema 6 15": "NEMA615", "nema 6 15r": "NEMA615",
    "nema 14 50": "NEMA1450", "nema 14 50p": "NEMA1450",
    "schuko cee7 4": "SCHUKO", "cee 7 4 schuko": "SCHUKO",
    "type e french": "TYPEE", "cee 7 5": "TYPEE",
}

# «Se cobra» o «no se cobra». OpenStreetMap usa la clave fee (yes/no).
MAPA_COBRO = {
    "no": "GRATIS", "free": "GRATIS", "gratis": "GRATIS",
    "gratuito": "GRATIS", "0": "GRATIS",
    "no pago": "GRATIS", "sin pago": "GRATIS", "sin costo": "GRATIS",
    "yes": "COBRA", "si": "COBRA",
}

# Condición de uso. Cuatro niveles, no tres:
#   PUBLICO            cualquiera puede cargar
#   PUBLICO_MEMBRESIA  público, pero exige cuenta, tarjeta o aviso previo
#   CLIENTES           solo clientes, visitantes o personal del sitio
#   PRIVADO            cerrado
#
# Corrección respecto de la versión anterior: «customers» (OSM) y «Private -
# For Staff, Visitors or Customers» (OCM) describen lo mismo y caían en
# códigos distintos (PUBLICO_RESTRINGIDO y PRIVADO). Ahora ambos son CLIENTES.
MAPA_ACCESO = {
    "yes": "PUBLICO", "public": "PUBLICO", "permissive": "PUBLICO",
    "publico": "PUBLICO", "public pay at location": "PUBLICO",
    "public membership required": "PUBLICO_MEMBRESIA",
    "public notice required": "PUBLICO_MEMBRESIA",
    "customers": "CLIENTES", "destination": "CLIENTES",
    "private for staff visitors or customers": "CLIENTES",
    "private for staff and visitors": "CLIENTES",
    "private": "PRIVADO", "privately owned notice required": "PRIVADO",
    "private restricted access": "PRIVADO",
    "no": "PRIVADO", "permit": "PRIVADO",
}

# OSM no codifica la membresía en access (va en authentication:*), de modo
# que access=yes no contradice «Public - Membership Required»: dice menos,
# no dice otra cosa. Contarlo como desacuerdo mide granularidad, no conflicto.
ACCESO_COMPATIBLE = {frozenset({"PUBLICO", "PUBLICO_MEMBRESIA"})}


def acceso_compatible(a, b):
    return a == b or frozenset({a, b}) in ACCESO_COMPATIBLE


def conectores_osm(sockets_json):
    """Conjunto de conectores declarados en sockets_json, más los no mapeados.
    Se recorren las claves del diccionario, no su representación en texto, y
    se ignoran las subclaves (socket:<tipo>:output, :current…), que describen
    un conector ya contado y no uno nuevo."""
    out, sin = set(), set()
    for k, v in a_diccionario(sockets_json).items():
        partes = k.split(":")
        if len(partes) != 2 or partes[0] != "socket":
            continue
        if str(v).strip().lower() in ("0", "no"):
            continue
        tipo = partes[1].lower()
        c = MAPA_CONECTOR.get(tipo)
        out.add(c) if c else sin.add(tipo)
    return out, sin


def conectores_ocm(lista):
    if not declarado(lista):
        return set(), set()
    out, sin = set(), set()
    for p in re.split(r"[;|]", str(lista)):
        p = p.strip()
        if not p or not declarado(p):
            continue
        c = MAPA_CONECTOR.get(normalizar(p))
        out.add(c) if c else sin.add(p)
    return out, sin


def traducir(valor, mapa):
    """Traduce un valor al vocabulario común. Devuelve (código, sin_equivalencia)."""
    if not declarado(valor):
        return None, None
    c = mapa.get(normalizar(valor))
    return (c, None) if c else (None, str(valor))


def traducir_cobro(valor):
    """costo_uso de OpenChargeMap es texto libre: unas veces trae una categoría
    («Free») y otras la tarifa en pesos («1650/kWh», «COP 1690 kWh»). Un valor
    numérico distinto de cero significa que la estación cobra. Tratarlo como
    campo categórico contaría cada tarifa como un término sin equivalencia,
    cuando todas significan lo mismo."""
    if not declarado(valor):
        return None, None
    n = normalizar(valor)
    if n in MAPA_COBRO:
        return MAPA_COBRO[n], None
    x = numero(n)
    if x is not None:
        return ("GRATIS" if x == 0 else "COBRA"), None
    return None, str(valor)

# Carga de atributos y de los pares de B3

In [5]:
CIUDADES = ("Bogota", "Medellin")


def cargar_atributos():
    """Atributos completos de cada estación, indexados por su identificador.

    Los identificadores se fijan como texto: son etiquetas, no cantidades.
    Con nulos presentes pandas los infiere como float y escribe '475409.0',
    que no une contra '475409'. La versión anterior lo resolvía con
    str(int(...)), que además revienta si el valor viene vacío.
    """
    osm, ocm = {}, {}
    for ciudad in CIUDADES:
        ruta_osm = constantes.exigir_archivo(
            RUTA_A2 / f"A2_estaciones_OSM_{ciudad}.geojson")
        with open(ruta_osm, encoding="utf-8") as fh:
            for feature in json.load(fh)["features"]:
                p = dict(feature["properties"])
                p["_ciudad"] = ciudad
                uid = str(p["osm_uid"])
                if uid in osm:
                    raise ValueError(f"osm_uid repetido entre ciudades: {uid}")
                osm[uid] = p

        ruta_ocm = constantes.exigir_archivo(
            RUTA_A3 / f"A3_OpenChargeMap_{ciudad}_estaciones.csv")
        datos = pd.read_csv(ruta_ocm, encoding="utf-8-sig",
                            dtype={"ocm_id": "string"})
        datos["ocm_id"] = (datos["ocm_id"].str.strip()
                           .str.replace(r"\.0$", "", regex=True))
        if datos["ocm_id"].isna().any():
            raise ValueError(
                f"{ciudad}: hay registros de OpenChargeMap sin ocm_id en "
                f"{ruta_ocm.name}. El identificador es obligatorio.")
        for _, fila in datos.iterrows():
            fila = fila.copy()
            fila["_ciudad"] = ciudad
            if fila.ocm_id in ocm:
                raise ValueError(f"ocm_id repetido entre ciudades: {fila.ocm_id}")
            ocm[fila.ocm_id] = fila
    return osm, ocm


OSM_AT, OCM_AT = cargar_atributos()

# --- Pares de B3 -------------------------------------------------------------
# Se leen con la función compartida y no con un read_csv suelto. Esa función
# fija los identificadores como texto y LANZA ERROR si el número de pares no
# coincide con el que declara B3_parametros.json. La versión anterior de B4
# corrió con 12 pares cuando B3 ya tenía 13, y nada lo detectó.
PARES, _CANDIDATOS_B3, PARAM_B3 = constantes.cargar_resultados_b3()

huerfanos_osm = sorted(set(PARES.osm_uid) - set(OSM_AT))
huerfanos_ocm = sorted(set(PARES.ocm_id) - set(OCM_AT))
if huerfanos_osm or huerfanos_ocm:
    raise KeyError(
        "Pares de B3 que apuntan a estaciones ausentes de A2/A3 — "
        f"OSM: {huerfanos_osm} · OCM: {huerfanos_ocm}. B3 y A2/A3 vienen de "
        "versiones distintas.")

print(f"Estaciones con atributos: OSM {len(OSM_AT)} | OCM {len(OCM_AT)}")
print(f"Pares emparejados por B3: {len(PARES)} "
      f"(umbral {PARAM_B3['decision_umbral']['umbral_adoptado']})")

# --- Alias de operador -------------------------------------------------------
# Se reutiliza el diccionario de B3: allí se verificó con fuentes que Enel X,
# Codensa y EVX designan la misma red, y que Voltex es la marca de
# electrolineras de Terpel. B4 no agrega alias propios: hacerlo crearía dos
# versiones del diccionario que divergen.
ALIAS_TABLA = pd.read_csv(
    constantes.exigir_archivo(RUTA_B3 / "B3_diccionario_alias_operador.csv"),
    encoding="utf-8-sig")
if ALIAS_TABLA["alias"].duplicated().any():
    raise ValueError("El diccionario de alias de B3 tiene alias repetidos.")
ALIAS_OPERADOR = dict(ALIAS_TABLA[["alias", "identificador"]].values)


def resolver_operador(valor):
    """Identificador común del operador, o None si no hay dato o no hay alias."""
    if not declarado(valor):
        return None
    return ALIAS_OPERADOR.get(normalizar(valor))


print(f"Alias de operador cargados de B3: {len(ALIAS_OPERADOR)} "
      f"({ALIAS_TABLA['identificador'].nunique()} operadores)")

# --- Términos sin equivalencia -----------------------------------------------
# Toda lectura pasa por valor_osm() / valor_ocm(), la misma que usan las
# secciones siguientes. Si aquí se leyera distinto, este inventario de
# términos describiría otros datos que los que se comparan después.
sin_mapear = {"conector": set(), "cobro": set(), "acceso": set(),
              "operador": set()}

for m in OCM_AT.values():
    sin_mapear["conector"] |= conectores_ocm(valor_ocm(m, "conectores_lista"))[1]
    _, s = traducir_cobro(valor_ocm(m, "costo_uso"))
    if s:
        sin_mapear["cobro"].add(s)
    _, s = traducir(valor_ocm(m, "tipo_uso"), MAPA_ACCESO)
    if s:
        sin_mapear["acceso"].add(s)
    op = valor_ocm(m, "operador")
    if declarado(op) and resolver_operador(op) is None:
        sin_mapear["operador"].add(str(op))

for o in OSM_AT.values():
    sin_mapear["conector"] |= conectores_osm(valor_osm(o, "sockets_json"))[1]
    _, s = traducir_cobro(valor_osm(o, "cobro"))
    if s:
        sin_mapear["cobro"].add(s)
    _, s = traducir(valor_osm(o, "acceso"), MAPA_ACCESO)
    if s:
        sin_mapear["acceso"].add(s)
    op = valor_osm(o, "operador")
    if declarado(op) and resolver_operador(op) is None:
        sin_mapear["operador"].add(str(op))

print("\n--- Términos sin equivalencia en el vocabulario ---")
for k, v in sin_mapear.items():
    print(f"  {k:10s} {sorted(v) if v else 'ninguno'}")
print("\nLos operadores sin alias son operadores reales sin equivalencia")
print("conocida: se dejan sin resolver en vez de inventarles una.")

# --- Verificación independiente: operador resuelto igual que en B3 -----------
# B3 escribió en op_osm / op_ocm el identificador que resolvió para cada par,
# por su propio camino (cargar_fuentes_estaciones + su normalización). Si B4
# resuelve distinto, la tabla de concordancia de operador no sería comparable
# con el emparejamiento que la originó.
DISCREPANCIAS_OPERADOR_B3 = []
for r in PARES.itertuples():
    a = resolver_operador(valor_osm(OSM_AT[r.osm_uid], "operador"))
    b = resolver_operador(valor_ocm(OCM_AT[r.ocm_id], "operador"))
    esperado_a = r.op_osm if pd.notna(r.op_osm) else None
    esperado_b = r.op_ocm if pd.notna(r.op_ocm) else None
    if (a, b) != (esperado_a, esperado_b):
        DISCREPANCIAS_OPERADOR_B3.append(
            {"ocm_id": r.ocm_id, "osm_uid": r.osm_uid,
             "b4_osm": a, "b3_osm": esperado_a,
             "b4_ocm": b, "b3_ocm": esperado_b})

print(f"\nOperador resuelto igual que B3 en "
      f"{len(PARES) - len(DISCREPANCIAS_OPERADOR_B3)} de {len(PARES)} pares.")
if DISCREPANCIAS_OPERADOR_B3:
    print(pd.DataFrame(DISCREPANCIAS_OPERADOR_B3).to_string(index=False))

Estaciones con atributos: OSM 53 | OCM 38
Pares emparejados por B3: 13 (umbral 0.5)
Alias de operador cargados de B3: 21 (8 operadores)

--- Términos sin equivalencia en el vocabulario ---
  conector   ninguno
  cobro      ninguno
  acceso     ninguno
  operador   ['Car-B', 'Hotel Tequendama', 'Isagen', 'Tesla', 'Tesla (Tesla-only charging)']

Los operadores sin alias son operadores reales sin equivalencia
conocida: se dejan sin resolver en vez de inventarles una.

Operador resuelto igual que B3 en 13 de 13 pares.


# Sección 1: tabla de concordancia por atributo

In [6]:
# =============================================================================
# Sección 1 — Tabla de concordancia por atributo
#
# valor_osm() y valor_ocm() ya NO se definen aquí: viven en la celda de
# utilidades, porque la celda de carga las necesita antes. Redefinirlas en
# esta celda crearía dos versiones de la misma regla de lectura.
# =============================================================================

# Tolerancia para la potencia: las fuentes redondean distinto (7 frente a
# 7.2 kW en el mismo cargador). Un kilovatio no cambia la clase de carga
# (lenta, semirrápida, rápida), que es lo que importa para el ruteo.
TOLERANCIA_POTENCIA_KW = 1.0


def comparar(nombre, v_osm, v_ocm):
    """Compara un atributo entre fuentes sobre el vocabulario común.

    Devuelve un dict con:
      concuerda  True / False, o None si alguna fuente no declara. None NO es
                 desacuerdo: es ausencia de dato.
      jaccard    solo para conector (proporción de conectores compartidos).
      detalle    texto legible con los valores originales y su traducción.
    """
    vacio = {"concuerda": None, "jaccard": None}

    if nombre == "conector":
        a, _ = conectores_osm(v_osm)
        b, _ = conectores_ocm(v_ocm)
        if not a or not b:
            return {**vacio, "detalle": "alguna fuente no declara"}
        j = len(a & b) / len(a | b)
        return {"concuerda": a == b, "jaccard": round(j, 3),
                "detalle": f"OSM {sorted(a)} · OCM {sorted(b)} · Jaccard {j:.2f}"}

    if nombre == "potencia":
        a, b = numero(v_osm), numero(v_ocm)
        if a is None or b is None:
            return {**vacio, "detalle": "alguna fuente no declara"}
        return {"concuerda": abs(a - b) <= TOLERANCIA_POTENCIA_KW, "jaccard": None,
                "detalle": f"OSM {a:g} kW · OCM {b:g} kW"}

    if nombre == "capacidad":
        # Como cantidad, no como cadena: «2» y «2.0» son el mismo valor.
        a, b = numero(v_osm), numero(v_ocm)
        if a is None or b is None:
            return {**vacio, "detalle": "alguna fuente no declara"}
        return {"concuerda": a == b, "jaccard": None,
                "detalle": f"OSM {a:g} · OCM {b:g}"}

    if nombre == "cobro":
        ca, _ = traducir_cobro(v_osm)
        cb, _ = traducir_cobro(v_ocm)
        if ca is None or cb is None:
            return {**vacio, "detalle": "alguna fuente no declara o usa un término sin equivalencia"}
        return {"concuerda": ca == cb, "jaccard": None,
                "detalle": f"OSM «{v_osm}» → {ca} · OCM «{v_ocm}» → {cb}"}

    if nombre == "acceso":
        # access=yes en OSM no contradice «Public - Membership Required»: OSM
        # codifica la membresía en otra etiqueta. Se compara por
        # compatibilidad, no por igualdad estricta.
        ca, _ = traducir(v_osm, MAPA_ACCESO)
        cb, _ = traducir(v_ocm, MAPA_ACCESO)
        if ca is None or cb is None:
            return {**vacio, "detalle": "alguna fuente no declara o usa un término sin equivalencia"}
        return {"concuerda": acceso_compatible(ca, cb), "jaccard": None,
                "detalle": f"OSM «{v_osm}» → {ca} · OCM «{v_ocm}» → {cb}"}

    if nombre == "operador":
        ia, ib = resolver_operador(v_osm), resolver_operador(v_ocm)
        if ia is None or ib is None:
            return {**vacio, "detalle": "alguna fuente no declara o no resuelve a un alias"}
        return {"concuerda": ia == ib, "jaccard": None,
                "detalle": f"OSM «{v_osm}» → {ia} · OCM «{v_ocm}» → {ib}"}

    raise ValueError(f"Atributo sin regla de comparación: {nombre!r}")


filas = []
for r in PARES.itertuples():
    o, m = OSM_AT[r.osm_uid], OCM_AT[r.ocm_id]
    for nombre, k_osm, k_ocm, _ in ATRIBUTOS:
        vo, vm = valor_osm(o, k_osm), valor_ocm(m, k_ocm)
        res = comparar(nombre, vo, vm)
        filas.append({
            "atributo": nombre, "ciudad": r.ciudad,
            "ocm_id": r.ocm_id, "osm_uid": r.osm_uid,
            "nombre_ocm": r.nombre_ocm,
            "declara_osm": declarado(vo), "declara_ocm": declarado(vm),
            "comparable": res["concuerda"] is not None,
            "concuerda": res["concuerda"],
            "jaccard_conector": res["jaccard"],
            "detalle": res["detalle"],
        })

CONC = pd.DataFrame(filas)
# Tipo booleano con nulos: distingue False (desacuerdo) de <NA> (sin dato).
# Con object, fillna(False) confundía las dos cosas al contar.
CONC["concuerda"] = CONC["concuerda"].astype("boolean")
for col in ("ocm_id", "osm_uid"):
    CONC[col] = CONC[col].astype("string")

print("=" * 78)
print(f"SECCIÓN 1 · CONCORDANCIA POR ATRIBUTO  (sobre {len(PARES)} pares de B3)")
print("=" * 78)
T1 = (CONC.groupby("atributo")
      .agg(declara_osm=("declara_osm", "sum"),
           declara_ocm=("declara_ocm", "sum"),
           comparables=("comparable", "sum"),
           concuerdan=("concuerda", lambda s: int((s == True).sum())),
           discrepan=("concuerda", lambda s: int((s == False).sum())))
      .reindex([a[0] for a in ATRIBUTOS]))
T1["n_pares"] = len(PARES)
print(T1.to_string())

print("\n--- Casos comparables, uno por uno ---")
C = CONC[CONC.comparable]
if len(C):
    for _, r in C.iterrows():
        marca = "=" if r.concuerda else "≠"
        print(f"  [{marca}] {r.atributo:10s} {r.ocm_id:>7} · {str(r.nombre_ocm)[:34]:34s}")
        print(f"          {r.detalle}")
else:
    print("  (ninguno)")

n_total = len(CONC)
n_c = int(CONC.comparable.sum())
n_ok = int((CONC.concuerda == True).sum())
n_no = int((CONC.concuerda == False).sum())
print(f"\nCombinaciones par-atributo: {n_total}")
print(f"  comparables:  {n_c} ({100*n_c/n_total:.0f} %)")
print(f"    concuerdan: {n_ok}")
print(f"    discrepan:  {n_no}")
print(f"  no comparables: {n_total - n_c} — en estas alguna fuente no declara el")
print("  atributo. No son desacuerdos: son ausencias de declaración.")

# --- Pares sin ningún conector en común --------------------------------------
# Un desacuerdo parcial de conector (Jaccard entre 0 y 1) es compatible con que
# las fuentes describan la misma estación con distinto detalle. Jaccard = 0
# significa que no comparten NINGÚN conector, lo que es evidencia de atributos
# contra la identidad del par. No se reabre B3 —la arquitectura es lineal—:
# se reporta como hallazgo para el informe.
SIN_CONECTOR_COMUN = CONC[(CONC.atributo == "conector")
                          & (CONC.jaccard_conector == 0)]
print("\n--- Pares de B3 sin ningún conector en común ---")
if len(SIN_CONECTOR_COMUN):
    for _, r in SIN_CONECTOR_COMUN.iterrows():
        par = PARES[(PARES.ocm_id == r.ocm_id) & (PARES.osm_uid == r.osm_uid)].iloc[0]
        print(f"  {r.ocm_id} ↔ {r.osm_uid}  «{r.nombre_ocm}»")
        print(f"      {r.detalle}")
        print(f"      distancia {par.distancia_m:.1f} m · puntaje {par.puntaje:.3f} · "
              f"componentes: {par.componentes}")
    print("  Evidencia contra la identidad del par. Se reporta; B3 no se modifica.")
else:
    print("  ninguno")

SECCIÓN 1 · CONCORDANCIA POR ATRIBUTO  (sobre 13 pares de B3)
           declara_osm  declara_ocm  comparables  concuerdan  discrepan  n_pares
atributo                                                                        
potencia             0           10            0           0          0       13
conector             3           13            3           1          2       13
acceso               1           13            1           1          0       13
cobro                2           12            2           2          0       13
capacidad            2           13            2           2          0       13
operador             9            8            6           6          0       13

--- Casos comparables, uno por uno ---
  [≠] conector     85062 · Centro comercial Unicentro Bogotá 
          OSM ['CCS1', 'CHADEMO', 'NEMA515', 'TYPE1', 'TYPE2'] · OCM ['CCS2', 'CHADEMO', 'TYPE1', 'TYPE2'] · Jaccard 0.50
  [=] cobro        85062 · Centro comercial Unicentro Bogotá 
    

# Sección 2: completitud por fuente

In [7]:
# Universo: los inventarios de A2 y A3 tal como vienen (53 OSM, 38 OCM), sin
# emparejar ni deduplicar. Esta sección describe a cada FUENTE; la cobertura
# del inventario consolidado es la sección 4.
#
# Toda lectura pasa por valor_osm() / valor_ocm(). En particular, la potencia
# de OSM incluye la declarada por conector (socket:*:output) y el operador
# incluye marca y red. La versión anterior leía solo la clave general y
# reportaba 0 de 53 estaciones OSM con potencia.
# =============================================================================

print("=" * 78)
print("SECCIÓN 2 · COMPLETITUD DE ATRIBUTOS POR FUENTE")
print("=" * 78)

filas = []
for nombre, k_osm, k_ocm, desc in ATRIBUTOS:
    for fuente, universo, extractor, clave in [
            ("OSM", OSM_AT, valor_osm, k_osm),
            ("OCM", OCM_AT, valor_ocm, k_ocm)]:
        for ciudad in CIUDADES:
            sub = [v for v in universo.values() if v["_ciudad"] == ciudad]
            n = len(sub)
            d = sum(1 for v in sub if declarado(extractor(v, clave)))
            filas.append({"atributo": nombre, "fuente": fuente,
                          "ciudad": ciudad, "declaran": d, "total": n,
                          "pct": round(100 * d / n, 1) if n else np.nan})

COMPL = pd.DataFrame(filas)

# Conteos absolutos junto al porcentaje: «85 %» no dice nada sin saber sobre
# cuántas estaciones (regla 4 del equipo).
TAB2 = (COMPL
        .assign(celda=lambda t: t.declaran.astype(str) + "/" + t.total.astype(str)
                + " (" + t.pct.map(lambda x: f"{x:.0f}") + " %)")
        .pivot(index="atributo", columns=["fuente", "ciudad"], values="celda")
        .reindex([a[0] for a in ATRIBUTOS]))
print(TAB2.to_string())

print("\n--- Lectura, ambas ciudades ---")
for nombre, _, _, desc in ATRIBUTOS:
    s = COMPL[(COMPL.atributo == nombre) & (COMPL.fuente == "OSM")]
    o = COMPL[(COMPL.atributo == nombre) & (COMPL.fuente == "OCM")]
    d, n = int(s.declaran.sum()), int(s.total.sum())
    do, no = int(o.declaran.sum()), int(o.total.sum())
    print(f"  {nombre:10s} OSM {d:2d}/{n:2d} ({100*d/n:5.1f} %) · "
          f"OCM {do:2d}/{no:2d} ({100*do/no:5.1f} %)   {desc}")

print("\nNota: «operador» cuenta si la fuente DECLARA un valor, no si ese valor")
print("resuelve a un alias conocido. B3 midió la resolución (B3_cobertura_operador.csv),")
print("que es menor. Se declaran pero no resuelven: "
      + ", ".join(sorted(sin_mapear["operador"])) + ".")

# --- Verificación independiente: potencia en OSM -----------------------------
# potencia_osm() lee la columna 'potencia' y el diccionario sockets_json. Aquí
# se recuenta por otro camino: directamente sobre tags_json, que es la copia
# completa de las etiquetas que devolvió Overpass, buscando cualquier clave
# que declare potencia. Si los dos conteos difieren, A2 perdió una clave al
# aplanar las etiquetas, o potencia_osm() está leyendo mal.
CLAVES_POTENCIA_OSM = ("charging_station:output", "maxpower", "power")


def declara_potencia_en_tags(o):
    tags = a_diccionario(o.get("tags_json"))
    return any(k in CLAVES_POTENCIA_OSM
               or (k.startswith("socket:") and k.endswith(":output"))
               for k in tags)


VERIF_POTENCIA_OSM = {}
for ciudad in CIUDADES:
    sub = [o for o in OSM_AT.values() if o["_ciudad"] == ciudad]
    por_tags = sum(declara_potencia_en_tags(o) for o in sub)
    por_funcion = int(COMPL[(COMPL.atributo == "potencia")
                            & (COMPL.fuente == "OSM")
                            & (COMPL.ciudad == ciudad)].declaran.iloc[0])
    VERIF_POTENCIA_OSM[ciudad] = (por_funcion, por_tags)
    estado = "coincide" if por_funcion == por_tags else "NO COINCIDE"
    print(f"\nPotencia OSM {ciudad}: potencia_osm() {por_funcion} · "
          f"tags_json {por_tags} → {estado}")

if any(a != b for a, b in VERIF_POTENCIA_OSM.values()):
    raise RuntimeError(
        "El conteo de potencia en OSM difiere entre potencia_osm() y tags_json. "
        "Revisar qué clave de potencia no está leyendo la función.")

SECCIÓN 2 · COMPLETITUD DE ATRIBUTOS POR FUENTE
fuente              OSM                         OCM               
ciudad           Bogota     Medellin         Bogota       Medellin
atributo                                                          
potencia     1/42 (2 %)   0/11 (0 %)   23/27 (85 %)  11/11 (100 %)
conector   11/42 (26 %)   0/11 (0 %)  27/27 (100 %)  11/11 (100 %)
acceso       2/42 (5 %)   0/11 (0 %)  27/27 (100 %)  11/11 (100 %)
cobro       5/42 (12 %)   0/11 (0 %)   19/27 (70 %)    9/11 (82 %)
capacidad   4/42 (10 %)   1/11 (9 %)  27/27 (100 %)  11/11 (100 %)
operador   19/42 (45 %)  5/11 (46 %)   19/27 (70 %)    6/11 (54 %)

--- Lectura, ambas ciudades ---
  potencia   OSM  1/53 (  1.9 %) · OCM 34/38 ( 89.5 %)   kW disponibles; determina el tiempo de carga
  conector   OSM 11/53 ( 20.8 %) · OCM 38/38 (100.0 %)   compatibilidad física con el vehículo
  acceso     OSM  2/53 (  3.8 %) · OCM 38/38 (100.0 %)   si la estación es de uso público
  cobro      OSM  5/53 (  9.4

# Sección 3: dirección del enriquecimiento

In [8]:
# Para cada atributo, en cuántos pares de B3 una fuente declara un valor que
# la otra no tiene. Mide si la reconciliación es simétrica.
#
# Correcciones respecto de la versión anterior:
#   - Lee con valor_osm() / valor_ocm(). Antes hacía o.get(k_osm) directo, de
#     modo que el operador de OSM ignoraba marca y red aunque las secciones 1
#     y 2 ya los leyeran: tres secciones, dos reglas.
#   - La conclusión se construye a partir de los conteos. Antes afirmaba
#     «casi exclusivamente de OCM hacia OSM» con texto fijo, que seguiría
#     impreso aunque los datos dijeran otra cosa.
# =============================================================================

print("=" * 78)
print("SECCIÓN 3 · DIRECCIÓN DEL ENRIQUECIMIENTO")
print("=" * 78)
print(f"Sobre los {len(PARES)} pares de B3: en cuántos una fuente declara un valor")
print("que la otra no tiene.\n")

COLUMNAS_DIR = ["ambas_declaran", "OCM_llena_a_OSM",
                "OSM_llena_a_OCM", "ninguna_declara"]

filas = []
for nombre, k_osm, k_ocm, _ in ATRIBUTOS:
    conteo = dict.fromkeys(COLUMNAS_DIR, 0)
    for r in PARES.itertuples():
        a = declarado(valor_osm(OSM_AT[r.osm_uid], k_osm))
        b = declarado(valor_ocm(OCM_AT[r.ocm_id], k_ocm))
        if a and b:
            conteo["ambas_declaran"] += 1
        elif b:
            conteo["OCM_llena_a_OSM"] += 1
        elif a:
            conteo["OSM_llena_a_OCM"] += 1
        else:
            conteo["ninguna_declara"] += 1
    filas.append({"atributo": nombre, **conteo})

DIR = pd.DataFrame(filas).set_index("atributo")[COLUMNAS_DIR]
print(DIR.to_string())

# --- Verificación cruzada con la sección 1 -----------------------------------
# Las dos secciones leen los mismos pares por caminos distintos (aquí se cuenta
# por combinación de declaraciones; allá se construyó fila a fila). Si la regla
# de lectura divergiera, estos totales dejarían de cuadrar.
_s1 = CONC.groupby("atributo")[["declara_osm", "declara_ocm"]].sum()
_s3 = pd.DataFrame({
    "declara_osm": DIR.ambas_declaran + DIR.OSM_llena_a_OCM,
    "declara_ocm": DIR.ambas_declaran + DIR.OCM_llena_a_OSM,
})
if not _s1.reindex(_s3.index).astype(int).compare(_s3.astype(int)).empty:
    raise RuntimeError(
        "Las declaraciones por atributo no cuadran entre la sección 1 y la 3: "
        "las dos secciones están leyendo los atributos con reglas distintas.")
print("\nDeclaraciones por atributo: cuadran con la sección 1.")

# --- Conclusión construida desde los datos -----------------------------------
tot_ocm = int(DIR.OCM_llena_a_OSM.sum())
tot_osm = int(DIR.OSM_llena_a_OCM.sum())
aporta_osm = DIR.index[DIR.OSM_llena_a_OCM > 0].tolist()
aporta_ocm = DIR.index[DIR.OCM_llena_a_OSM > 0].tolist()

print(f"\nHuecos que OpenChargeMap llena en OpenStreetMap: {tot_ocm} "
      f"(en {len(aporta_ocm)} de {len(ATRIBUTOS)} atributos)")
print(f"Huecos que OpenStreetMap llena en OpenChargeMap: {tot_osm} "
      f"(en {len(aporta_osm)} de {len(ATRIBUTOS)} atributos)")

if tot_osm == 0 and tot_ocm == 0:
    print("\nNinguna fuente completa a la otra en estos pares.")
elif tot_osm == 0:
    print("\nEl flujo es unidireccional: solo OpenChargeMap aporta atributos.")
elif tot_ocm == 0:
    print("\nEl flujo es unidireccional: solo OpenStreetMap aporta atributos.")
else:
    mayor, menor = ("OpenChargeMap", "OpenStreetMap") if tot_ocm >= tot_osm \
        else ("OpenStreetMap", "OpenChargeMap")
    aporte_menor = aporta_osm if mayor == "OpenChargeMap" else aporta_ocm
    print(f"\n{mayor} aporta {max(tot_ocm, tot_osm)} huecos por cada "
          f"{min(tot_ocm, tot_osm)} que aporta {menor}.")
    print(f"{menor} solo aporta en: {', '.join(aporte_menor)}.")

SECCIÓN 3 · DIRECCIÓN DEL ENRIQUECIMIENTO
Sobre los 13 pares de B3: en cuántos una fuente declara un valor
que la otra no tiene.

           ambas_declaran  OCM_llena_a_OSM  OSM_llena_a_OCM  ninguna_declara
atributo                                                                    
potencia                0               10                0                3
conector                3               10                0                0
acceso                  1               12                0                0
cobro                   2               10                0                1
capacidad               2               11                0                0
operador                6                2                3                2

Declaraciones por atributo: cuadran con la sección 1.

Huecos que OpenChargeMap llena en OpenStreetMap: 55 (en 6 de 6 atributos)
Huecos que OpenStreetMap llena en OpenChargeMap: 3 (en 1 de 6 atributos)

OpenChargeMap aporta 55 huecos por cada 3 que apor

# Sección 4: cobertura del inventario para el ruteo

In [9]:
# El inventario se toma de B2.1, que es la clasificación vigente con los
# veredictos aplicados. B3 no produce inventario: solo pares.
#
# Tres correcciones respecto de la versión anterior:
#
# 1. SE CUENTA SOBRE ESTACIONES, NO SOBRE REGISTROS. B2.1 tiene 74 registros
#    y 71 estaciones. Las 3 filas C5 (punto secundario del mismo sitio) repiten
#    el ocm_id de su par principal, así que contarlas sumaba dos veces los
#    atributos de Unicentro, Compensar y San Andresito. Las 74 filas se
#    conservan en el CSV de salida; las proporciones se miden sobre 71.
#
# 2. SE MARCA QUÉ PARES NO SOSTIENE B3. B2.1 tiene 15 pares y B3 sostiene 13.
#    En los 2 restantes, la sección asigna a una estación OSM los atributos de
#    una OCM que el puntaje combinado no acepta como la misma. No se reabre
#    nada hacia atrás: se agrega la columna par_sostenido_por_b3 y se mide qué
#    cambia si esos 2 pares se separan.
#
# 3. Los identificadores se leen como texto, y los atributos con
#    valor_osm() / valor_ocm(), como en el resto del cuaderno.
# =============================================================================

print("=" * 78)
print("SECCIÓN 4 · COBERTURA DEL INVENTARIO CONSOLIDADO PARA EL RUTEO")
print("=" * 78)

ARCH_INV = constantes.exigir_archivo(RUTA_B2_1 / "B2_1_clasificacion_verificada.csv")
INV = pd.read_csv(ARCH_INV, encoding="utf-8-sig",
                  dtype={"ocm_id": "string", "osm_uid": "string", "caso": "string"})
INV["ocm_id"] = INV["ocm_id"].str.strip().str.replace(r"\.0$", "", regex=True)
N_REGISTROS_B2_1 = len(INV)
INV = INV[INV.osm_uid.notna() | INV.ocm_id.notna()].copy()

INV["es_secundario"] = INV["categoria"].str.startswith("C5")

# --- ¿B3 sostiene el par? ----------------------------------------------------
# Solo tiene sentido en filas principales con las dos fuentes. En el resto
# queda <NA>: no hay par que sostener.
PARES_B3_SET = set(zip(PARES.ocm_id, PARES.osm_uid))
es_par = INV.ocm_id.notna() & INV.osm_uid.notna() & ~INV.es_secundario
INV["par_sostenido_por_b3"] = pd.Series(pd.NA, index=INV.index, dtype="boolean")
INV.loc[es_par, "par_sostenido_por_b3"] = [
    (a, b) in PARES_B3_SET
    for a, b in zip(INV.loc[es_par, "ocm_id"], INV.loc[es_par, "osm_uid"])]


def atributo_efectivo(ocm_id, osm_uid, k_osm, k_ocm):
    """Fuente de la que se toma el atributo tras consolidar.
    Se prefiere OCM cuando las dos declaran, porque es la fuente especializada
    y la sección 2 muestra que declara casi todo. Devuelve None si ninguna."""
    if pd.notna(ocm_id):
        m = OCM_AT.get(ocm_id)
        if m is not None and declarado(valor_ocm(m, k_ocm)):
            return "OCM"
    if pd.notna(osm_uid):
        o = OSM_AT.get(osm_uid)
        if o is not None and declarado(valor_osm(o, k_osm)):
            return "OSM"
    return None


# Las filas C5 no reciben origen: su estación ya está contada en la fila
# principal. Quedan en <NA> en el CSV para que nadie las sume por error.
for nombre, k_osm, k_ocm, _ in ATRIBUTOS:
    INV[f"origen_{nombre}"] = pd.array(
        [None if sec else atributo_efectivo(a, b, k_osm, k_ocm)
         for a, b, sec in zip(INV.ocm_id, INV.osm_uid, INV.es_secundario)],
        dtype="string")

EST = INV[~INV.es_secundario].copy()

# --- Unicidad: cada estación una sola vez ------------------------------------
dup_ocm = EST.ocm_id.dropna()[EST.ocm_id.dropna().duplicated()].tolist()
dup_osm = EST.osm_uid.dropna()[EST.osm_uid.dropna().duplicated()].tolist()
if dup_ocm or dup_osm:
    raise RuntimeError(
        f"Hay estaciones repetidas fuera de las filas C5 — OCM: {dup_ocm} · "
        f"OSM: {dup_osm}. La cobertura contaría atributos dos veces.")

print(f"Registros de B2.1: {N_REGISTROS_B2_1} · "
      f"filas C5 (secundarias): {int(INV.es_secundario.sum())} · "
      f"estaciones: {len(EST)}")
print("  por ciudad: " + " · ".join(f"{c} {n}" for c, n in
                                    EST.groupby("ciudad").size().items()))

# --- Cobertura por atributo --------------------------------------------------
filas = []
for nombre, _, _, desc in ATRIBUTOS:
    col = EST[f"origen_{nombre}"]
    filas.append({"atributo": nombre, "estaciones": len(EST),
                  "disponible": int(col.notna().sum()),
                  "de_OCM": int((col == "OCM").sum()),
                  "de_OSM": int((col == "OSM").sum()),
                  "ausente": int(col.isna().sum())})
COBERTURA = pd.DataFrame(filas).set_index("atributo")
COBERTURA["pct_disponible"] = (100 * COBERTURA.disponible
                               / COBERTURA.estaciones).round(1)
print()
print(COBERTURA.to_string())

# --- Verificación independiente de la potencia -------------------------------
# Por otro camino: conjuntos de identificadores con potencia, construidos
# directamente sobre las fuentes (OCM: columna potencia_kw_max; OSM: tags_json,
# la misma lectura independiente de la sección 2), y cruzados contra EST. No
# pasa por atributo_efectivo() ni por valor_osm().
IDS_OCM_CON_POTENCIA = {k for k, m in OCM_AT.items()
                        if numero(m.potencia_kw_max) is not None}
IDS_OSM_CON_POTENCIA = {k for k, o in OSM_AT.items()
                        if declara_potencia_en_tags(o)}
con_potencia_indep = int(sum(
    (pd.notna(a) and a in IDS_OCM_CON_POTENCIA)
    or (pd.notna(b) and b in IDS_OSM_CON_POTENCIA)
    for a, b in zip(EST.ocm_id, EST.osm_uid)))
if con_potencia_indep != COBERTURA.loc["potencia", "disponible"]:
    raise RuntimeError(
        f"Potencia disponible: atributo_efectivo() da "
        f"{COBERTURA.loc['potencia', 'disponible']} y el cruce por conjuntos "
        f"da {con_potencia_indep}.")
print(f"\nPotencia disponible verificada por cruce de conjuntos: {con_potencia_indep}.")

# --- Sensibilidad a los pares que B3 no sostiene -----------------------------
# Si esos pares se separan, cada uno pasa a ser dos estaciones y cada lado
# conserva solo sus propios atributos.
NO_SOSTENIDO = EST.par_sostenido_por_b3.eq(False).fillna(False).astype(bool)
PARES_NO_SOSTENIDOS = EST[NO_SOSTENIDO]

print(f"\n--- Pares de B2.1 que B3 no sostiene: {len(PARES_NO_SOSTENIDOS)} ---")
for r in PARES_NO_SOSTENIDOS.itertuples():
    print(f"  {r.ocm_id} «{r.nombre_ocm}» ↔ {r.osm_uid} «{r.nombre_osm}» "
          f"({r.distancia_m:.1f} m)")

filas = []
for nombre, k_osm, k_ocm, _ in ATRIBUTOS:
    base = int(EST.loc[~NO_SOSTENIDO, f"origen_{nombre}"].notna().sum())
    separados = sum(
        (atributo_efectivo(r.ocm_id, pd.NA, k_osm, k_ocm) is not None)
        + (atributo_efectivo(pd.NA, r.osm_uid, k_osm, k_ocm) is not None)
        for r in PARES_NO_SOSTENIDOS.itertuples())
    n_sep = len(EST) + len(PARES_NO_SOSTENIDOS)
    filas.append({
        "atributo": nombre,
        "disponible_B2_1": int(COBERTURA.loc[nombre, "disponible"]),
        "estaciones_B2_1": len(EST),
        "disponible_separando": base + separados,
        "estaciones_separando": n_sep,
    })
SENSIB_PARES = pd.DataFrame(filas).set_index("atributo")
print()
print(SENSIB_PARES.to_string())

# --- Restricción para el objetivo específico 5 -------------------------------
sin_pot = EST[EST.origen_potencia.isna()]
print("\n--- Restricción para el objetivo específico 5 ---")
print(f"Estaciones sin potencia declarada: {len(sin_pot)} de {len(EST)} "
      f"({100*len(sin_pot)/len(EST):.0f} %)")
fila_s = SENSIB_PARES.loc["potencia"]
sin_sep = fila_s.estaciones_separando - fila_s.disponible_separando
print(f"Separando los pares que B3 no sostiene: {sin_sep} de "
      f"{fila_s.estaciones_separando} ({100*sin_sep/fila_s.estaciones_separando:.0f} %)")
print("Sin potencia no puede estimarse el tiempo de carga: esas estaciones")
print("entran al grafo como nodos de ubicación conocida y capacidad desconocida.")
print()
print(sin_pot.groupby(["ciudad", "categoria"]).size().to_string())

SECCIÓN 4 · COBERTURA DEL INVENTARIO CONSOLIDADO PARA EL RUTEO
Registros de B2.1: 74 · filas C5 (secundarias): 3 · estaciones: 71
  por ciudad: Bogota 52 · Medellin 19

           estaciones  disponible  de_OCM  de_OSM  ausente  pct_disponible
atributo                                                                  
potencia           71          34      33       1       37            47.9
conector           71          45      37       8       26            63.4
acceso             71          38      37       1       33            53.5
cobro              71          31      28       3       40            43.7
capacidad          71          40      37       3       31            56.3
operador           71          42      25      17       29            59.2

Potencia disponible verificada por cruce de conjuntos: 34.

--- Pares de B2.1 que B3 no sostiene: 2 ---
  475122 «Terpel av Boyacá» ↔ node/13009418551 «Punto de carga Voltex Avenida Boyacá» (79.5 m)
  470940 «EnelX San Andresito» 

# Pruebas

In [10]:
_pares_b2_1 = set(zip(INV.loc[es_par, "ocm_id"], INV.loc[es_par, "osm_uid"]))

# Cruce del número de estaciones por identidad, sin usar la columna categoria:
# estaciones = OCM distintas + OSM distintas − pares principales − filas C5
# (cada par principal junta una OCM y una OSM en una estación; cada C5 agrega
# un OSM que pertenece a una estación ya contada).
_n_ocm = INV.ocm_id.dropna().nunique()
_n_osm = INV.osm_uid.dropna().nunique()
_n_pares_principales = int(es_par.sum())
_n_c5 = int(INV.es_secundario.sum())
_estaciones_por_identidad = _n_ocm + _n_osm - _n_pares_principales - _n_c5

_tesla = potencia_osm(OSM_AT.get("node/13995048783", {}))
_conect = conectores_osm({"socket:type2": "1",
                          "socket:type2:output": "22 kW",
                          "socket:chademo": "0"})[0]
_acc_osm = traducir("customers", MAPA_ACCESO)[0]
_acc_ocm = traducir("Private - For Staff, Visitors or Customers", MAPA_ACCESO)[0]
_bolera = CONC[(CONC.atributo == "conector") & (CONC.ocm_id == "75628")]

PRUEBAS = [
    # --- CADENA --------------------------------------------------------------
    ("cadena", "pares de B3 = los declarados en B3_parametros.json",
     len(PARES) == PARAM_B3["decision_umbral"]["pares_resultantes"],
     f"{len(PARES)} frente a {PARAM_B3['decision_umbral']['pares_resultantes']}"),
    ("cadena", "todo par de B3 existe como par en B2.1",
     PARES_B3_SET <= _pares_b2_1,
     f"{len(PARES_B3_SET - _pares_b2_1)} pares de B3 ausentes de B2.1"),
    ("cadena", "B4 no descarta registros de B2.1",
     len(INV) == N_REGISTROS_B2_1,
     f"{len(INV)} de {N_REGISTROS_B2_1}"),

    # --- CRUCE ---------------------------------------------------------------
    ("cruce", "operador resuelto igual que B3 (op_osm / op_ocm)",
     len(DISCREPANCIAS_OPERADOR_B3) == 0,
     f"{len(DISCREPANCIAS_OPERADOR_B3)} discrepancias"),
    ("cruce", "potencia OSM: potencia_osm() = tags_json, por ciudad",
     all(a == b for a, b in VERIF_POTENCIA_OSM.values()),
     str(VERIF_POTENCIA_OSM)),
    ("cruce", "estaciones por categoría = estaciones por identidad",
     len(EST) == _estaciones_por_identidad,
     f"{len(EST)} frente a {_n_ocm}+{_n_osm}−{_n_pares_principales}−{_n_c5}"
     f"={_estaciones_por_identidad}"),
    ("cruce", "potencia disponible: atributo_efectivo() = cruce de conjuntos",
     con_potencia_indep == int(COBERTURA.loc["potencia", "disponible"]),
     f"{con_potencia_indep}"),
    ("cruce", "toda fila C5 repite el ocm_id de una estación contada",
     set(INV.loc[INV.es_secundario, "ocm_id"].dropna()) <= set(EST.ocm_id.dropna()),
     ""),
    ("cruce", "dirección del enriquecimiento suma el total de pares",
     bool((DIR.sum(axis=1) == len(PARES)).all()),
     ""),
    ("cruce", "una fila por par y atributo en la concordancia",
     len(CONC) == len(PARES) * len(ATRIBUTOS),
     f"{len(CONC)} filas"),

    # --- ANCLA ---------------------------------------------------------------
    ("ancla", "Tesla Bogotá (node/13995048783) declara 325 kW por conector",
     _tesla == 325.0,
     str(_tesla)),
    ("ancla", "conectores_osm ignora subclaves :output y cantidad 0",
     _conect == {"TYPE2"},
     str(sorted(_conect))),
    ("ancla", "«customers» y «Private - For Staff, Visitors or Customers» → mismo código",
     _acc_osm is not None and _acc_osm == _acc_ocm,
     f"{_acc_osm} · {_acc_ocm}"),
    ("ancla", "tarifa 0 es GRATIS y tarifa en pesos es COBRA",
     traducir_cobro("0")[0] == "GRATIS"
     and traducir_cobro("COP 1690 kWh")[0] == "COBRA",
     ""),
    ("ancla", "Bolera Salitre (75628) sin ningún conector en común",
     len(_bolera) == 1 and _bolera.jaccard_conector.iloc[0] == 0,
     ""),
    ("ancla", "ningún conector, cobro ni acceso sin equivalencia",
     not (sin_mapear["conector"] or sin_mapear["cobro"] or sin_mapear["acceso"]),
     str({k: sorted(v) for k, v in sin_mapear.items() if k != "operador" and v})),
]

RESUMEN_PRUEBAS = pd.DataFrame(PRUEBAS, columns=["clase", "prueba", "pasa", "detalle"])
TODAS_PASAN = bool(RESUMEN_PRUEBAS.pasa.all())

print("=" * 78)
print("PRUEBAS DE ACEPTACIÓN")
print("=" * 78)
for clase, nombre, pasa, detalle in PRUEBAS:
    print(f"  [{'OK ' if pasa else 'FALLA'}] {clase:6s} {nombre}")
    if detalle and not pasa:
        print(f"                 {detalle}")
print(f"\n  {int(RESUMEN_PRUEBAS.pasa.sum())} de {len(PRUEBAS)} pasan · "
      f"{'TODAS PASAN' if TODAS_PASAN else 'HAY FALLAS — revisar antes de reportar'}")

PRUEBAS DE ACEPTACIÓN
  [OK ] cadena pares de B3 = los declarados en B3_parametros.json
  [OK ] cadena todo par de B3 existe como par en B2.1
  [OK ] cadena B4 no descarta registros de B2.1
  [OK ] cruce  operador resuelto igual que B3 (op_osm / op_ocm)
  [OK ] cruce  potencia OSM: potencia_osm() = tags_json, por ciudad
  [OK ] cruce  estaciones por categoría = estaciones por identidad
  [OK ] cruce  potencia disponible: atributo_efectivo() = cruce de conjuntos
  [OK ] cruce  toda fila C5 repite el ocm_id de una estación contada
  [OK ] cruce  dirección del enriquecimiento suma el total de pares
  [OK ] cruce  una fila por par y atributo en la concordancia
  [OK ] ancla  Tesla Bogotá (node/13995048783) declara 325 kW por conector
  [OK ] ancla  conectores_osm ignora subclaves :output y cantidad 0
  [OK ] ancla  «customers» y «Private - For Staff, Visitors or Customers» → mismo código
  [OK ] ancla  tarifa 0 es GRATIS y tarifa en pesos es COBRA
  [OK ] ancla  Bolera Salitre (75628) sin 

# Guardado

In [11]:
print("=" * 78)
print("GUARDADO")
print("=" * 78)

# --- 1. Tablas ---------------------------------------------------------------
SALIDAS_CSV = {
    # Sección 1 — el entregable pedido
    "B4_concordancia_por_atributo.csv":        (CONC, False),
    "B4_resumen_concordancia.csv":             (T1, True),
    "B4_pares_sin_conector_comun.csv":         (SIN_CONECTOR_COMUN, False),
    # Sección 2
    "B4_completitud_por_fuente.csv":           (COMPL, False),
    # Sección 3
    "B4_direccion_enriquecimiento.csv":        (DIR, True),
    # Sección 4 — el inventario conserva sus 74 filas; la cobertura se mide
    # sobre las 71 estaciones y va en su propia tabla
    "B4_inventario_atributos_ruteo.csv":       (INV, False),
    "B4_cobertura_por_atributo.csv":           (COBERTURA, True),
    "B4_sensibilidad_pares_no_sostenidos.csv": (SENSIB_PARES, True),
    # Verificación
    "B4_pruebas_aceptacion.csv":               (RESUMEN_PRUEBAS, False),
}


def como_texto(tabla):
    """Representación en texto para comparar lo escrito con lo releído.
    Se compara el contenido, no los tipos: .equals() compara también dtypes y
    en B2.1 produjo dos fallos falsos (object frente a string tras releer)."""
    t = tabla.reset_index(drop=False) if tabla.index.name else tabla.reset_index(drop=True)
    return t.astype("string").fillna("").reset_index(drop=True)


RELECTURA = {}
for nombre, (tabla, con_indice) in SALIDAS_CSV.items():
    ruta = RUTA_OUT / nombre
    tabla.to_csv(ruta, index=con_indice, encoding="utf-8-sig")
    releida = pd.read_csv(ruta, encoding="utf-8-sig", dtype=str,
                          keep_default_na=False,
                          index_col=0 if con_indice else None)
    a, b = como_texto(tabla), como_texto(releida)
    RELECTURA[nombre] = (list(a.columns) == list(b.columns)
                         and a.shape == b.shape
                         and a.compare(b).empty)
    print(f"  {nombre:42s} {len(tabla):>4d} filas · "
          f"relectura {'OK' if RELECTURA[nombre] else 'FALLA'}")

if not all(RELECTURA.values()):
    raise RuntimeError(
        "Relectura fallida en: "
        + ", ".join(k for k, v in RELECTURA.items() if not v)
        + ". Lo que quedó en disco no es lo que calculó el cuaderno.")

# --- 2. Vocabularios ---------------------------------------------------------
with open(RUTA_OUT / "B4_vocabularios.json", "w", encoding="utf-8") as fh:
    json.dump({
        "conector": MAPA_CONECTOR,
        "cobro": MAPA_COBRO,
        "acceso": MAPA_ACCESO,
        "acceso_compatible": sorted(sorted(par) for par in ACCESO_COMPATIBLE),
        "centinelas": sorted(constantes.CENTINELAS),
        "sin_equivalencia": {k: sorted(v) for k, v in sin_mapear.items()},
    }, fh, ensure_ascii=False, indent=2, sort_keys=True)
print(f"  {'B4_vocabularios.json':42s}")

# --- 3. Parámetros, conteos y limitaciones -----------------------------------
_pot = COBERTURA.loc["potencia"]
_sens = SENSIB_PARES.loc["potencia"]

PARAMETROS = {
    "bloque": "B4",
    "proposito": ("reconciliar atributos de ruteo entre OpenStreetMap y "
                  "OpenChargeMap y medir su completitud"),
    "fecha_corte_inventario": constantes.FECHA_CORTE_LOCAL,

    "insumos": {
        "pares": "B3_pares_puntaje_combinado.csv",
        "umbral_B3": PARAM_B3["decision_umbral"]["umbral_adoptado"],
        "inventario": "B2_1_clasificacion_verificada.csv",
        "alias_operador": "B3_diccionario_alias_operador.csv",
    },

    "reglas_de_lectura": {
        "potencia_osm": ("clave general; si falta, máximo de socket:*:output"),
        "operador_osm": "operador, marca o red, en ese orden (misma regla que B3)",
        "atributo_consolidado": "se toma de OCM si declara; si no, de OSM",
        "tolerancia_potencia_kw": TOLERANCIA_POTENCIA_KW,
        "acceso": ("PUBLICO y PUBLICO_MEMBRESIA se tratan como compatibles: "
                   "OSM no codifica la membresía en access"),
    },

    "resultados": {
        "pares_evaluados": len(PARES),
        "combinaciones_par_atributo": int(len(CONC)),
        "comparables": int(CONC.comparable.sum()),
        "concuerdan": int((CONC.concuerda == True).sum()),
        "discrepan": int((CONC.concuerda == False).sum()),
        "pares_sin_conector_comun": SIN_CONECTOR_COMUN.ocm_id.tolist(),
        "huecos_OCM_llena_a_OSM": int(DIR.OCM_llena_a_OSM.sum()),
        "huecos_OSM_llena_a_OCM": int(DIR.OSM_llena_a_OCM.sum()),
        "registros_inventario": int(len(INV)),
        "estaciones_inventario": int(len(EST)),
        "estaciones_con_potencia": int(_pot.disponible),
        "estaciones_sin_potencia": int(_pot.ausente),
        "potencia_osm_por_ciudad": {c: v[0] for c, v in VERIF_POTENCIA_OSM.items()},
        "pares_B2_1_no_sostenidos_por_B3": PARES_NO_SOSTENIDOS.ocm_id.tolist(),
        "sin_potencia_separando_no_sostenidos": {
            "sin_potencia": int(_sens.estaciones_separando - _sens.disponible_separando),
            "estaciones": int(_sens.estaciones_separando),
        },
    },

    "limitaciones_declaradas": [
        (f"la concordancia se mide sobre {len(PARES)} pares: describe esos "
         "pares, no sostiene afirmaciones sobre las fuentes en general"),
        (f"B3 no sostiene {len(PARES_NO_SOSTENIDOS)} de los "
         f"{int(es_par.sum())} pares de B2.1; la cobertura se reporta con "
         "ambos criterios (ver B4_sensibilidad_pares_no_sostenidos.csv)"),
        ("las filas C5 no aportan atributos propios: su estación se cuenta "
         "en la fila principal"),
        ("los operadores declarados sin alias ("
         + ", ".join(sorted(sin_mapear["operador"]))
         + ") cuentan como declarados pero no son comparables"),
    ],

    "atribucion": [constantes.ATRIBUCION_ODBL, constantes.ATRIBUCION_OCM],
    "pruebas_pasan": TODAS_PASAN,
    "pruebas": f"{int(RESUMEN_PRUEBAS.pasa.sum())} de {len(RESUMEN_PRUEBAS)}",
}

with open(RUTA_OUT / "B4_parametros.json", "w", encoding="utf-8") as fh:
    json.dump(PARAMETROS, fh, ensure_ascii=False, indent=2)
print(f"  {'B4_parametros.json':42s}")

# --- 4. Manifiesto de insumos ------------------------------------------------
# Mismo formato plano de B2, B2.1 y B3: nombre de archivo -> sha256, más las
# entradas 'crudo::' heredadas sin recalcular.
MANIFIESTO = dict(HASHES_INSUMOS)
MANIFIESTO.update(CRUDOS_HEREDADOS)
with open(RUTA_OUT / "B4_manifiesto_insumos.json", "w", encoding="utf-8") as fh:
    json.dump(MANIFIESTO, fh, ensure_ascii=False, indent=2, sort_keys=True)
print(f"  {'B4_manifiesto_insumos.json':42s} {len(MANIFIESTO)} entradas")

# --- 5. Huellas de las salidas -----------------------------------------------
HUELLAS = {n: constantes.sha256_archivo(RUTA_OUT / n) for n in SALIDAS_CSV}
for n in ("B4_vocabularios.json", "B4_parametros.json"):
    HUELLAS[n] = constantes.sha256_archivo(RUTA_OUT / n)
with open(RUTA_OUT / "B4_huellas_salidas.json", "w", encoding="utf-8") as fh:
    json.dump(HUELLAS, fh, ensure_ascii=False, indent=2, sort_keys=True)
print(f"  {'B4_huellas_salidas.json':42s} {len(HUELLAS)} archivos")

print(f"\n  Archivos en: {RUTA_OUT}")
print(f"  Pruebas de aceptación: "
      f"{'TODAS PASAN' if TODAS_PASAN else 'HAY FALLAS — revisar antes de reportar'}")
print("  Las salidas de A2, A3, B2.1 y B3 quedan intactas.")

GUARDADO
  B4_concordancia_por_atributo.csv             78 filas · relectura OK
  B4_resumen_concordancia.csv                   6 filas · relectura OK
  B4_pares_sin_conector_comun.csv               1 filas · relectura OK
  B4_completitud_por_fuente.csv                24 filas · relectura OK
  B4_direccion_enriquecimiento.csv              6 filas · relectura OK
  B4_inventario_atributos_ruteo.csv            74 filas · relectura OK
  B4_cobertura_por_atributo.csv                 6 filas · relectura OK
  B4_sensibilidad_pares_no_sostenidos.csv       6 filas · relectura OK
  B4_pruebas_aceptacion.csv                    16 filas · relectura OK
  B4_vocabularios.json                      
  B4_parametros.json                        
  B4_manifiesto_insumos.json                 11 entradas
  B4_huellas_salidas.json                    11 archivos

  Archivos en: F:\Tesis\Bloque B\Entregable B4
  Pruebas de aceptación: TODAS PASAN
  Las salidas de A2, A3, B2.1 y B3 quedan intactas.
